# Log-Uniform Initialization: MNIST Classification & 2D PIELM

Two experiments to strengthen the paper:
1. **MNIST/FashionMNIST ELM** — Log-Uniform vs Gaussian vs Uniform for classification (not just PDEs)
2. **2D PIELM** — Poisson, Helmholtz, Multi-Freq on 2D domains

Both use the same three initialization strategies from the paper.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import math
import os

os.makedirs('draft_figures', exist_ok=True)

dtype = torch.float64
device = torch.device('cpu')

# ── Weight Initialization (same as paper) ──

def init_weights(input_dim, hidden_dim, init_type='power', scale=30.0,
                 alpha=1.0, dtype=torch.float64):
    """Initialize weights for ELM/PIELM.
    
    init_type:
      'power'  : log-uniform on [1, scale], i.e. |w| = 10^U(0, log10(scale))
      'normal' : Gaussian N(0, scale)
      'uniform': Uniform U(-scale, scale)
    """
    if init_type == 'power':
        # Log-uniform: |w| = 10^u, u ~ U(0, log10(scale))
        u = torch.rand(input_dim, hidden_dim, dtype=dtype)
        magnitudes = 10 ** (u * math.log10(max(scale, 1.01)))
        signs = 2 * torch.randint(0, 2, (input_dim, hidden_dim)).to(dtype) - 1
        W = signs * magnitudes
        b = (2 * torch.rand(1, hidden_dim, dtype=dtype) - 1) * math.pi
    elif init_type == 'normal':
        W = torch.randn(input_dim, hidden_dim, dtype=dtype) * scale
        b = torch.randn(1, hidden_dim, dtype=dtype) * scale
    elif init_type == 'uniform':
        W = (2 * torch.rand(input_dim, hidden_dim, dtype=dtype) - 1) * scale
        b = (2 * torch.rand(1, hidden_dim, dtype=dtype) - 1) * scale
    else:
        raise ValueError(f'Unknown init_type: {init_type}')
    return W, b

INIT_TYPES = ['power', 'normal', 'uniform']
LABELS = {'power': 'Log-Uniform', 'normal': 'Gaussian', 'uniform': 'Uniform'}
COLORS = {'power': '#2ca02c', 'normal': '#1f77b4', 'uniform': '#d62728'}

print('Setup complete.')

---
## Part 1: MNIST & FashionMNIST Classification

ELM for classification: `H = sigmoid(X @ W + b)`, solve `beta = (H'H + λI)^-1 H'Y`.

Compare Log-Uniform vs Gaussian vs Uniform initialization for W and b.
All use the same hidden dimension, same regularization, same solver.

In [ ]:
import torchvision
import torchvision.transforms as transforms

def load_dataset(name='MNIST'):
    transform = transforms.Compose([transforms.ToTensor()])
    if name == 'MNIST':
        train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else:
        train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    
    x_train = train.data.view(-1, 784).float() / 255.0
    y_train = torch.nn.functional.one_hot(train.targets, 10).float()
    x_test = test.data.view(-1, 784).float() / 255.0
    y_test = test.targets
    return x_train, y_train, x_test, y_test


def run_elm_classification(x_train, y_train, x_test, y_test,
                           init_type, hidden_dim=1024, scale=1.0,
                           activation='sigmoid', lambd=1e-2, seed=0):
    """Run ELM classification with given init type. Returns accuracy and time."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    input_dim = x_train.shape[1]
    
    # Initialize weights
    if init_type == 'power':
        u = torch.rand(input_dim, hidden_dim)
        magnitudes = 10 ** (u * math.log10(max(scale, 1.01)))
        signs = 2 * torch.randint(0, 2, (input_dim, hidden_dim)).float() - 1
        W = signs * magnitudes
        b = (2 * torch.rand(1, hidden_dim) - 1) * math.pi
    elif init_type == 'normal':
        W = torch.randn(input_dim, hidden_dim) * scale
        b = torch.randn(1, hidden_dim) * scale
    elif init_type == 'uniform':
        W = (2 * torch.rand(input_dim, hidden_dim) - 1) * scale
        b = (2 * torch.rand(1, hidden_dim) - 1) * scale
    
    start = time.time()
    
    # Feature matrix
    if activation == 'sigmoid':
        H_train = torch.sigmoid(x_train @ W + b)
    elif activation == 'sin':
        H_train = torch.sin(x_train @ W + b)
    elif activation == 'relu':
        H_train = torch.relu(x_train @ W + b)
    
    # Solve: beta = (H'H + lambda*I)^-1 H'Y
    I = torch.eye(hidden_dim)
    HTH = H_train.t() @ H_train
    HTY = H_train.t() @ y_train
    beta = torch.linalg.solve(HTH + I / lambd, HTY)
    
    elapsed = time.time() - start
    
    # Evaluate
    if activation == 'sigmoid':
        H_test = torch.sigmoid(x_test @ W + b)
    elif activation == 'sin':
        H_test = torch.sin(x_test @ W + b)
    elif activation == 'relu':
        H_test = torch.relu(x_test @ W + b)
    
    y_pred = (H_test @ beta).argmax(dim=1)
    acc = (y_pred == y_test).float().mean().item() * 100
    
    return acc, elapsed

print('Classification functions ready.')

In [ ]:
# ── Experiment: MNIST & FashionMNIST ──
# Compare 3 init types × 2 datasets × multiple seeds × scale sweep

N_SEEDS = 10
H_DIMS = [256, 512, 1024, 2048]
SCALES_CLASS = [0.5, 1.0, 2.0, 5.0, 10.0]
DATASETS = ['MNIST', 'FashionMNIST']
ACTIVATION = 'sigmoid'

results_class = {}

for ds_name in DATASETS:
    print(f'\n{"=" * 60}')
    print(f'Dataset: {ds_name}')
    print(f'{"=" * 60}')
    x_train, y_train, x_test, y_test = load_dataset(ds_name)
    results_class[ds_name] = {}
    
    for h_dim in H_DIMS:
        results_class[ds_name][h_dim] = {}
        for init in INIT_TYPES:
            results_class[ds_name][h_dim][init] = {}
            for scale in SCALES_CLASS:
                accs = []
                times = []
                for seed in range(N_SEEDS):
                    acc, t = run_elm_classification(
                        x_train, y_train, x_test, y_test,
                        init_type=init, hidden_dim=h_dim,
                        scale=scale, activation=ACTIVATION,
                        seed=seed)
                    accs.append(acc)
                    times.append(t)
                results_class[ds_name][h_dim][init][scale] = {
                    'acc_mean': np.mean(accs),
                    'acc_std': np.std(accs),
                    'time_mean': np.mean(times),
                }
        
        # Print best scale for each init at this h_dim
        print(f'\nh={h_dim}:')
        for init in INIT_TYPES:
            best_scale = max(SCALES_CLASS,
                           key=lambda s: results_class[ds_name][h_dim][init][s]['acc_mean'])
            best = results_class[ds_name][h_dim][init][best_scale]
            print(f'  {LABELS[init]:<15s}: {best["acc_mean"]:.2f}% '
                  f'(±{best["acc_std"]:.2f}) at scale={best_scale}, '
                  f't={best["time_mean"]:.3f}s')

In [ ]:
# ── Classification Results: Accuracy vs Scale ──

fig, axes = plt.subplots(len(DATASETS), len(H_DIMS),
                         figsize=(4*len(H_DIMS), 4*len(DATASETS)),
                         sharey='row')
if len(DATASETS) == 1:
    axes = axes[np.newaxis, :]

for row, ds_name in enumerate(DATASETS):
    for col, h_dim in enumerate(H_DIMS):
        ax = axes[row, col]
        for init in INIT_TYPES:
            means = [results_class[ds_name][h_dim][init][s]['acc_mean']
                     for s in SCALES_CLASS]
            stds = [results_class[ds_name][h_dim][init][s]['acc_std']
                    for s in SCALES_CLASS]
            ax.errorbar(SCALES_CLASS, means, yerr=stds,
                       label=LABELS[init], color=COLORS[init],
                       marker='o', capsize=3)
        ax.set_xlabel('Weight Scale')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title(f'{ds_name}, h={h_dim}')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')

fig.suptitle('ELM Classification: Accuracy vs Weight Scale',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.savefig('draft_figures/mnist_scale_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to draft_figures/mnist_scale_sweep.png')

In [ ]:
# ── Classification Summary Table ──

print('\n' + '=' * 90)
print('CLASSIFICATION SUMMARY — Best scale per init type')
print('=' * 90)

for ds_name in DATASETS:
    print(f'\n{ds_name}:')
    print(f'{"h":<8s} {"Log-Uniform":<22s} {"Gaussian":<22s} {"Uniform":<22s}')
    print('-' * 80)
    for h_dim in H_DIMS:
        parts = [f'{h_dim:<8d}']
        for init in INIT_TYPES:
            best_s = max(SCALES_CLASS,
                        key=lambda s: results_class[ds_name][h_dim][init][s]['acc_mean'])
            r = results_class[ds_name][h_dim][init][best_s]
            parts.append(f'{r["acc_mean"]:.2f}±{r["acc_std"]:.2f} (s={best_s})')
        print(f'{parts[0]} {parts[1]:<22s} {parts[2]:<22s} {parts[3]:<22s}')

---
## Part 2: 2D PIELM Experiments

Extend the 1D PIELM framework to 2D domains.

For `φ(x,y) = sin(w1·x + w2·y + b)` with `W = [w1, w2]`:
- Laplacian: `Δφ = -(w1² + w2²)·φ`
- Helmholtz: `(-Δ - k²)φ = (w1² + w2² - k²)·φ`

In [ ]:
# ── 2D PIELM Solver ──

def make_2d_grid(n_per_dim, domain=(-1, 1, -1, 1)):
    """Create interior grid and boundary points for 2D square domain."""
    x0, x1, y0, y1 = domain
    x1d = torch.linspace(x0, x1, n_per_dim, dtype=dtype)
    y1d = torch.linspace(y0, y1, n_per_dim, dtype=dtype)
    xx, yy = torch.meshgrid(x1d, y1d, indexing='ij')
    X_int = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    
    # Boundary: 4 edges
    n_bc = n_per_dim
    bc_list = []
    for v in torch.linspace(x0, x1, n_bc, dtype=dtype):
        bc_list.append([v.item(), y0])
        bc_list.append([v.item(), y1])
    for v in torch.linspace(y0, y1, n_bc, dtype=dtype):
        bc_list.append([x0, v.item()])
        bc_list.append([x1, v.item()])
    X_bc = torch.tensor(bc_list, dtype=dtype).unique(dim=0)
    
    return X_int, X_bc


def run_pielm_2d(bench, init_type, scale, hidden_dim, n_per_dim,
                 seed, alpha=1.0, lambd=1e-10, bc_w=100.0):
    """Run one 2D PIELM trial."""
    torch.manual_seed(seed)
    
    domain = bench.get('domain', (-1, 1, -1, 1))
    X_int, X_bc = make_2d_grid(n_per_dim, domain)
    
    # Source and BCs
    f_int = bench['f_fn'](X_int)
    u_bc_vals = bench['u_fn'](X_bc)
    
    # Initialize weights: W: (2, h), b: (1, h)
    W, b = init_weights(2, hidden_dim, init_type, scale, alpha=alpha, dtype=dtype)
    
    start = time.time()
    
    # Feature matrix
    H_int = torch.sin(X_int @ W + b)   # (N_int, h)
    H_bc = torch.sin(X_bc @ W + b)     # (N_bc, h)
    
    # PDE operator
    w_sq_sum = (W ** 2).sum(dim=0, keepdim=True)  # (1, h): w1^2 + w2^2
    
    pde_type = bench.get('pde', 'poisson')
    if pde_type == 'poisson':
        H_pde = w_sq_sum * H_int           # -Δφ = (w1²+w2²)·φ
    elif pde_type == 'helmholtz':
        k = bench['k']
        H_pde = (w_sq_sum - k**2) * H_int  # (-Δ - k²)φ = (w1²+w2²-k²)·φ
    
    # Stack system
    A = torch.cat([H_pde, bc_w * H_bc], dim=0)
    rhs = torch.cat([f_int, bc_w * u_bc_vals], dim=0)
    
    # Solve
    ATA = A.t() @ A + lambd * torch.eye(hidden_dim, dtype=dtype)
    ATr = A.t() @ rhs
    beta = torch.linalg.solve(ATA, ATr)
    
    elapsed = time.time() - start
    
    # Evaluate on finer grid
    n_test = 50
    X_test, _ = make_2d_grid(n_test, domain)
    H_test = torch.sin(X_test @ W + b)
    u_pred = H_test @ beta
    u_true = bench['u_fn'](X_test)
    
    err = (torch.norm(u_pred - u_true) / torch.norm(u_true)).item()
    
    return dict(error=err, time=elapsed, u_pred=u_pred, u_true=u_true,
                X_test=X_test, n_test=n_test)


def run_multi_seed_2d(bench, init_type, scale, hidden_dim, n_per_dim, n_seeds):
    """Run multiple seeds and aggregate."""
    errors = []
    times = []
    last = None
    for seed in range(n_seeds):
        r = run_pielm_2d(bench, init_type, scale, hidden_dim, n_per_dim, seed)
        errors.append(r['error'])
        times.append(r['time'])
        last = r
    return dict(
        error_mean=np.mean(errors), error_std=np.std(errors),
        error_median=np.median(errors),
        time_mean=np.mean(times),
        errors=errors, last=last
    )

print('2D PIELM solver ready.')

In [ ]:
# ── 2D Benchmark Definitions ──

def make_poisson_2d_simple():
    """u(x,y) = sin(πx)·sin(πy)"""
    def u_fn(X):
        return (torch.sin(math.pi * X[:, 0:1]) *
                torch.sin(math.pi * X[:, 1:2]))
    def f_fn(X):
        return 2 * math.pi**2 * u_fn(X)
    return dict(name='Poisson 2D (1π,1π)', pde='poisson',
                f_fn=f_fn, u_fn=u_fn, domain=(-1,1,-1,1))


def make_poisson_2d_multifreq():
    """u(x,y) = sin(πx)·sin(2πy) + 0.3·sin(3πx)·sin(πy)"""
    def u_fn(X):
        x, y = X[:, 0:1], X[:, 1:2]
        return (torch.sin(math.pi*x) * torch.sin(2*math.pi*y) +
                0.3 * torch.sin(3*math.pi*x) * torch.sin(math.pi*y))
    def f_fn(X):
        x, y = X[:, 0:1], X[:, 1:2]
        # -Δu for each term: (w1² + w2²) * term
        return ((math.pi**2 + (2*math.pi)**2) *
                torch.sin(math.pi*x) * torch.sin(2*math.pi*y) +
                0.3 * ((3*math.pi)**2 + math.pi**2) *
                torch.sin(3*math.pi*x) * torch.sin(math.pi*y))
    return dict(name='Poisson 2D Multi-Freq', pde='poisson',
                f_fn=f_fn, u_fn=u_fn, domain=(-1,1,-1,1))


def make_helmholtz_2d(k_val=5.0):
    """Helmholtz 2D: -Δu - k²u = f, u = sin(πx)·sin(πy)"""
    def u_fn(X):
        return (torch.sin(math.pi * X[:, 0:1]) *
                torch.sin(math.pi * X[:, 1:2]))
    def f_fn(X):
        u = u_fn(X)
        return (2 * math.pi**2 - k_val**2) * u
    return dict(name=f'Helmholtz 2D (k={k_val})', pde='helmholtz',
                f_fn=f_fn, u_fn=u_fn, domain=(-1,1,-1,1), k=k_val)


def make_poisson_2d_highfreq():
    """u(x,y) = sin(πx)·sin(πy) + 0.2·sin(5πx)·sin(5πy)"""
    def u_fn(X):
        x, y = X[:, 0:1], X[:, 1:2]
        return (torch.sin(math.pi*x) * torch.sin(math.pi*y) +
                0.2 * torch.sin(5*math.pi*x) * torch.sin(5*math.pi*y))
    def f_fn(X):
        x, y = X[:, 0:1], X[:, 1:2]
        return (2*math.pi**2 * torch.sin(math.pi*x) * torch.sin(math.pi*y) +
                0.2 * 2*(5*math.pi)**2 *
                torch.sin(5*math.pi*x) * torch.sin(5*math.pi*y))
    return dict(name='Poisson 2D (1π+5π)', pde='poisson',
                f_fn=f_fn, u_fn=u_fn, domain=(-1,1,-1,1))


BENCHMARKS_2D = [
    make_poisson_2d_simple,
    make_poisson_2d_multifreq,
    make_helmholtz_2d,
    make_poisson_2d_highfreq,
]

print(f'{len(BENCHMARKS_2D)} 2D benchmarks defined.')

In [ ]:
# ── 2D Experiment 1: Benchmark at h=400 ──

N_SEEDS_2D = 20
H_DIM_2D = 400
SCALE_2D = 30.0
N_PER_DIM = 30  # 30x30 = 900 interior points

exp_2d_bench = {}

print(f'{"Problem":<30s} | {"Log-Uniform":<18s} {"Gaussian":<18s} {"Uniform":<18s} | Winner')
print('=' * 100)

for make_fn in BENCHMARKS_2D:
    bench = make_fn()
    name = bench['name']
    exp_2d_bench[name] = {}
    
    errs = {}
    for init in INIT_TYPES:
        r = run_multi_seed_2d(bench, init, SCALE_2D, H_DIM_2D, N_PER_DIM, N_SEEDS_2D)
        exp_2d_bench[name][init] = r
        errs[init] = r['error_mean']
    
    winner = min(errs, key=errs.get)
    print(f'{name:<30s} | {errs["power"]:<18.3e} {errs["normal"]:<18.3e} '
          f'{errs["uniform"]:<18.3e} | {LABELS[winner]}')

In [ ]:
# ── 2D Experiment 2: Neuron Budget Sweep ──

H_DIMS_2D = [20, 40, 60, 80, 100, 150, 200, 300, 400]
BUDGET_BENCHMARKS_2D = [make_poisson_2d_multifreq, make_helmholtz_2d,
                        make_poisson_2d_highfreq]

exp_2d_budget = {}

for make_fn in BUDGET_BENCHMARKS_2D:
    bench = make_fn()
    name = bench['name']
    exp_2d_budget[name] = {}
    print(f'\n--- {name} ---')
    
    for h in H_DIMS_2D:
        exp_2d_budget[name][h] = {}
        parts = [f'h={h:>4d}:']
        for init in INIT_TYPES:
            r = run_multi_seed_2d(bench, init, SCALE_2D, h, N_PER_DIM, N_SEEDS_2D)
            exp_2d_budget[name][h][init] = r
            parts.append(f'{LABELS[init]}={r["error_median"]:.2e}')
        
        ratio = exp_2d_budget[name][h]['normal']['error_median'] / \
                max(exp_2d_budget[name][h]['power']['error_median'], 1e-15)
        parts.append(f'G/PL={ratio:.1f}x')
        print('  '.join(parts))

In [ ]:
# ── 2D Budget Sweep Plots ──

fig, axes = plt.subplots(1, len(exp_2d_budget), figsize=(5*len(exp_2d_budget), 5))
if len(exp_2d_budget) == 1:
    axes = [axes]

for ax, (name, data) in zip(axes, exp_2d_budget.items()):
    for init in INIT_TYPES:
        hs = sorted(data.keys())
        medians = [data[h][init]['error_median'] for h in hs]
        means = [data[h][init]['error_mean'] for h in hs]
        stds = [data[h][init]['error_std'] for h in hs]
        ax.semilogy(hs, medians, '-o', color=COLORS[init],
                   label=LABELS[init], markersize=5)
        lower = [max(m - s, 1e-15) for m, s in zip(means, stds)]
        upper = [m + s for m, s in zip(means, stds)]
        ax.fill_between(hs, lower, upper, color=COLORS[init], alpha=0.15)
    
    # Annotate ratio at h=20
    if 20 in data:
        r = data[20]['normal']['error_median'] / max(data[20]['power']['error_median'], 1e-15)
        ax.annotate(f'{r:.0f}×', xy=(20, data[20]['power']['error_median']),
                   fontsize=11, fontweight='bold', color=COLORS['power'])
    
    ax.set_xlabel('Hidden Neurons (h)')
    ax.set_ylabel('L₂ Relative Error')
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('2D PIELM: Error vs Neuron Budget (Rm=30, 20 seeds)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.savefig('draft_figures/2d_neuron_budget.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to draft_figures/2d_neuron_budget.png')

In [ ]:
# ── 2D Solution Field Visualization ──

VIS_H = 40  # tight budget to show differences

fig, axes = plt.subplots(len(BENCHMARKS_2D), 4,
                         figsize=(20, 5*len(BENCHMARKS_2D)))

for row, make_fn in enumerate(BENCHMARKS_2D):
    bench = make_fn()
    
    # Exact solution
    r_exact = run_pielm_2d(bench, 'power', SCALE_2D, VIS_H, 30, seed=0)
    n = r_exact['n_test']
    X = r_exact['X_test']
    u_true = r_exact['u_true'].reshape(n, n).numpy()
    
    ax = axes[row, 0]
    im = ax.contourf(X[:, 0].reshape(n, n).numpy(),
                     X[:, 1].reshape(n, n).numpy(),
                     u_true, levels=20, cmap='RdBu_r')
    ax.set_title(f'{bench["name"]}\nExact', fontsize=10)
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax)
    
    for col, init in enumerate(INIT_TYPES):
        r = run_pielm_2d(bench, init, SCALE_2D, VIS_H, 30, seed=0)
        u_pred = r['u_pred'].reshape(n, n).numpy()
        err = r['error']
        
        ax = axes[row, col + 1]
        im = ax.contourf(X[:, 0].reshape(n, n).numpy(),
                         X[:, 1].reshape(n, n).numpy(),
                         u_pred, levels=20, cmap='RdBu_r')
        ax.set_title(f'{LABELS[init]}\nL₂={err:.2e}', fontsize=10)
        ax.set_aspect('equal')
        plt.colorbar(im, ax=ax)

fig.suptitle(f'2D PIELM Solution Fields at h={VIS_H}',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.savefig('draft_figures/2d_solution_fields.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to draft_figures/2d_solution_fields.png')

In [ ]:
# ── Summary ──

print('=' * 80)
print('SUMMARY')
print('=' * 80)

print('\n1. CLASSIFICATION (MNIST/FashionMNIST)')
print('   - Tests whether log-uniform helps for non-PDE tasks')
print('   - Compares 3 inits × multiple scales × multiple h values')

print('\n2. 2D PIELM')
print('   - Extends 1D findings to 2D domains')
print('   - Poisson, Multi-Freq Poisson, Helmholtz, High-Freq Poisson')
print('   - Budget sweep h=20..400')

print('\nKey question: Does the low-budget advantage persist in 2D?')
print('In 2D, each neuron has W=(w1,w2) — frequency is now a 2D vector.')
print('Log-uniform spreads |W| across magnitude decades, but frequency')
print('direction is random. At tight budgets, magnitude diversity should')
print('still help, but the 2D frequency coverage may require more neurons.')